### 1. Inisialisasi Environment & SparkSession
Mengunci runtime OpenJDK 11 terisolasi di Arch Linux dan membuat sesi komputasi PySpark

In [2]:
import os
import sys
import json
import getpass
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, when, round as spark_round

# Konfigurasi isolasi lingkungan native Arch Linux
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("Pertemuan6-Main-ArchLinux") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("=== STATUS KONEKSI SPARK ===")
print(f"User Host : {getpass.getuser()} | Spark : {spark.version} ({spark.sparkContext.master})")
print("Status    : SparkSession aktif & terisolasi penuh di Arch Linux!")

=== STATUS KONEKSI SPARK ===
User Host : kaky | Spark : 3.5.9 (local[*])
Status    : SparkSession aktif & terisolasi penuh di Arch Linux!


### 2. Komparasi Efisiensi Penyimpanan: CSV vs Parquet
Membangkitkan 20.000 data transaksi sintetis untuk membuktikan penghematan ukuran berkas berkat *columnar storage* dan kompresi Snappy

In [3]:
np.random.seed(11)
n = 20000
data_besar = {
    "order_id": [f"O{i}" for i in range(n)],
    "customer_id": np.random.randint(1, 21, size=n),
    "kategori": np.random.choice(["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"], size=n),
    "kota": np.random.choice(["Magelang", "Semarang", "Solo", "Yogyakarta", "Purworejo"], size=n),
    "metode_pembayaran": np.random.choice(["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"], size=n),
    "pendapatan": np.random.randint(50000, 500000, size=n),
}
pd.DataFrame(data_besar).to_csv("data_transaksi_besar.csv", index=False)
csv_size = os.path.getsize("data_transaksi_besar.csv")

# Tulis ulang sebagai Parquet
df_besar = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_besar.write.mode("overwrite").parquet("data_transaksi_besar_parquet")

parquet_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, fn in os.walk("data_transaksi_besar_parquet")
    for f in fn if f.endswith(".parquet")
)

print(f"Ukuran CSV mentah : {csv_size:,} bytes ({csv_size/1024:.1f} KB)")
print(f"Ukuran Parquet    : {parquet_size:,} bytes ({parquet_size/1024:.1f} KB)")
print(f"Efisiensi Ruang   : {(1 - parquet_size/csv_size)*100:.1f}% lebih hemat dibanding CSV!")

[Stage 2:>                                                          (0 + 1) / 1]

Ukuran CSV mentah : 892,909 bytes (872.0 KB)
Ukuran Parquet    : 202,197 bytes (197.5 KB)
Efisiensi Ruang   : 77.4% lebih hemat dibanding CSV!


### 3. Membaca Parquet (Metadata Skema Built-in)
Parquet menyimpan skema tipe data di internal metadata, sehingga pembacaan instan tanpa butuh `inferSchema=True`

In [4]:
df_dari_parquet = spark.read.parquet("data_transaksi_besar_parquet")
df_dari_parquet.printSchema()
df_dari_parquet.show(5)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)

+--------+-----------+----------+---------+-----------------+----------+
|order_id|customer_id|  kategori|     kota|metode_pembayaran|pendapatan|
+--------+-----------+----------+---------+-----------------+----------+
|      O0|         17|   Makanan|     Solo|              COD|    196650|
|      O1|         18| Kesehatan| Magelang|         E-Wallet|    283512|
|      O2|         14|Elektronik| Semarang|              COD|    108835|
|      O3|         13| Kesehatan|Purworejo|     Kartu Kredit|    390604|
|      O4|          2|   Fashion|Purworejo|         E-Wallet|    427224|
+--------+-----------+----------+---------+-----------------+----------+
only showing top 5 rows



### 4. Partitioning saat Menulis Data
Menulis Parquet terpartisi kolom `kota` untuk memicu optimasi *partition pruning* saat filtering wilayah.

In [4]:
df_besar.write.mode("overwrite").partitionBy("kota").parquet("data_transaksi_partisi_kota")
print("Struktur folder partisi kota:")
!ls -lh data_transaksi_partisi_kota

Struktur folder partisi kota:


Permissions Size User Date Modified Name
drwxr-xr-x     - kaky 30 Sep 21:31   kota=Magelang
drwxr-xr-x     - kaky 30 Sep 21:31   kota=Purworejo
drwxr-xr-x     - kaky 30 Sep 21:31   kota=Semarang
drwxr-xr-x     - kaky 30 Sep 21:31   kota=Solo
drwxr-xr-x     - kaky 30 Sep 21:31   kota=Yogyakarta
.rw-r--r--     8 kaky 30 Sep 21:31   ._SUCCESS.crc
.rw-r--r--     0 kaky 30 Sep 21:31  󰡯 _SUCCESS


### 5. Membaca Data JSON Lines (NDJSON)
Spark memproses data JSON per-baris mandiri (*JSON Lines*) agar dapat diproses paralel secara efisien.

In [5]:
np.random.seed(21)
pelanggan = [
    {
        "customer_id": i,
        "nama": f"Pelanggan{i}",
        "membership": np.random.choice(["Silver", "Gold", "Platinum"]),
        "kota_domisili": np.random.choice(["Magelang", "Semarang", "Solo", "Yogyakarta", "Purworejo"])
    }
    for i in range(1, 21)
]
with open("data_pelanggan.json", "w") as f:
    for p in pelanggan:
        f.write(json.dumps(p) + "\n")

df_pelanggan = spark.read.json("data_pelanggan.json")
df_pelanggan.printSchema()
df_pelanggan.show(5)

root
 |-- customer_id: long (nullable = true)
 |-- kota_domisili: string (nullable = true)
 |-- membership: string (nullable = true)
 |-- nama: string (nullable = true)



+-----------+-------------+----------+----------+
|customer_id|kota_domisili|membership|      nama|
+-----------+-------------+----------+----------+
|          1|     Magelang|      Gold|Pelanggan1|
|          2|     Magelang|    Silver|Pelanggan2|
|          3|     Magelang|    Silver|Pelanggan3|
|          4|         Solo|    Silver|Pelanggan4|
|          5|         Solo|      Gold|Pelanggan5|
+-----------+-------------+----------+----------+
only showing top 5 rows



### 6. Pipeline ETL Sederhana (Extract, Transform, Load)
Mengekstrak data transaksi dan pelanggan, menggabungkannya, menambahkan segmen transaksi, lalu menyimpan ke Parquet terpartisi `kategori`.

In [5]:
# EXTRACT
df_transaksi_etl = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_pelanggan_etl = spark.read.json("data_pelanggan.json")

# TRANSFORM
df_etl = df_transaksi_etl.join(df_pelanggan_etl, on="customer_id", how="left")
df_etl = df_etl.withColumn(
    "segmen_transaksi",
    when(col("pendapatan") >= 300000, "Tinggi")
    .when(col("pendapatan") >= 150000, "Sedang")
    .otherwise("Rendah")
)

# LOAD (Penyimpanan lokal & sink HDFS jika aktif)
local_etl_path = "data_transaksi_enriched_parquet"
df_etl.write.mode("overwrite").partitionBy("kategori").parquet(local_etl_path)
print(f"Hasil ETL tersimpan sukses di lokal: {local_etl_path}")

username = os.environ.get("USER", "kaky")
hdfs_etl_path = f"hdfs://localhost:9000/user/{username}/etl_output/transaksi_enriched"
try:
    os.system(f"hdfs dfs -mkdir -p /user/{username}/etl_output 2>/dev/null")
    df_etl.write.mode("overwrite").partitionBy("kategori").parquet(hdfs_etl_path)
    print(f"Hasil ETL tersimpan sukses di HDFS: {hdfs_etl_path}")
except Exception as e:
    print(f"HDFS offline saat runtime ({e.__class__.__name__}). Replikasi lokal terjaga.")

# Verifikasi membaca kembali
df_final = spark.read.parquet(local_etl_path)
print(f"\nVerifikasi hasil akhir: {df_final.count():,} baris utuh.")
df_final.groupBy("segmen_transaksi").agg(
    count("order_id").alias("jumlah_transaksi"),
    spark_round(avg("pendapatan"), 2).alias("rata_pendapatan")
).orderBy("segmen_transaksi").show()

Hasil ETL tersimpan sukses di lokal: data_transaksi_enriched_parquet


Hasil ETL tersimpan sukses di HDFS: hdfs://localhost:9000/user/kaky/etl_output/transaksi_enriched

Verifikasi hasil akhir: 20,000 baris utuh.
+----------------+----------------+---------------+
|segmen_transaksi|jumlah_transaksi|rata_pendapatan|
+----------------+----------------+---------------+
|          Rendah|            4464|       99639.28|
|          Sedang|            6633|      224445.06|
|          Tinggi|            8903|      398577.44|
+----------------+----------------+---------------+



### 8. Penutupan Sesi Spark
Membebaskan thread CPU dan memori JVM pada sistem host Linux.

In [8]:
spark.stop()
print("SparkSession berhasil dihentikan. Alokasi resource telah dibebaskan.")

SparkSession berhasil dihentikan. Alokasi resource telah dibebaskan.
